# ANÁLISE DE MACHINE LEARNING: Previsão de Sobrevivência no Titanic

**Autor:** Douglas Chaves Moura  
**Metodologia:** Análise Exploratória de Dados (EDA), Feature Engineering, Modelagem em Ensembles (Bagging e Boosting).

Este notebook apresenta uma solução analítica para a clássica competição de *Machine Learning* do Kaggle. O projeto vai muito além da aplicação cega de algoritmos: o objetivo é analisar os dados usando técnicas de **Exploração**, **Tratamento Estatístico (Missing Values, Scaling)** e **Otimização de Hiperparâmetros**.

Avaliaremos três abordagens metodológicas distintas de modelagem:
1. **Regressão Logística** (Modelo Linear Estocástico / Baseline)
2. **Random Forest** (Modelo de Ensemble - Bagging)
3. **XGBoost** (Modelo de Ensemble - Boosting)

## 1. IMPORTAÇÕES

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

import xgboost as xgb

# Ignorar warnings desnecessários
warnings.filterwarnings('ignore')

## 2. CARREGAMENTO DOS DADOS

In [ ]:
# Carregando os dados de treino e teste
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print("=== FORMA DOS DATASETS ===")
print(f"Treino: {train_df.shape}")
print(f"Teste: {test_df.shape}")

train_df.head()

## 3. ANÁLISE EXPLORATÓRIA DE DADOS (EDA)

In [ ]:
# 1. Preparação dos Dados: Cálculo de frequências e renomeação de categorias para o plot
surv_counts = train_df['Survived'].value_counts().reset_index()
surv_counts.columns = ['Survived', 'Count']
surv_counts['Survived'] = surv_counts['Survived'].map({0: 'Não Sobreviveu', 1: 'Sobreviveu'})

# 2. Criação da Figura: Gráfico de barras com mapeamento de cores personalizadas
fig_surv = px.bar(surv_counts, x='Survived', y='Count', 
                  title="Distribuição de Sobreviventes (Target)",
                  color='Survived',
                  color_discrete_map={'Não Sobreviveu': '#5E7E99', 'Sobreviveu': '#D4AF37'})

# 3. Rótulos e Interação: Formatação dos valores sobre as barras e do tooltip (hover)
fig_surv.update_traces(
    textposition="outside",
    texttemplate="%{y:,.0f}",
    hovertemplate="<b>%{x}</b><br>Quantidade: %{y:,.0f} passageiros<extra></extra>"
)

# 4. Estilização do Layout: Aplicação do tema escuro, tipografia e ajustes de eixo
fig_surv.update_layout(
    separators=",.",
    title=dict(x=0.5, font=dict(size=22, family="Arial", color="#E2E8F0")),
    xaxis_title="",
    yaxis_title="Quantidade de Passageiros",
    font=dict(family="Arial", size=14, color="#E2E8F0"),
    plot_bgcolor='#0B1320',
    paper_bgcolor='#0B1320',
    showlegend=False, # Legenda desativada pois o eixo X já identifica as classes
    xaxis=dict(showgrid=False, color="#E2E8F0"),
    yaxis=dict(showgrid=True, gridcolor="#1E293B", color="#E2E8F0", range=[0, surv_counts['Count'].max() * 1.20])
)

fig_surv.write_html('../site_port/graphics/titanic_fig_surv.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_surv.show()

### Interpretação: Desbalanceamento de Classes (Variável Target)

> **Insight:** Observamos um claro desbalanceamento na variável alvo (`Survived`). Cerca de 60% dos passageiros registrados neste conjunto de treinamento não sobreviveram ao naufrágio. O desbalanceamento exigirá cautela ao analisarmos métricas: em casos assim, focar apenas na **Acurácia** pode ser ilusório; métricas como **Precision**, **Recall** e **F1-Score** fornecerão uma avaliação de desempenho mais honesta.

In [ ]:
# 1. Preparação dos Dados: Criação de cópia isolada e mapeamento da variável alvo (Target) para texto explicativo
plot_df = train_df.copy()
plot_df['Status'] = plot_df['Survived'].map({0: 'Não Sobreviveu', 1: 'Sobreviveu'})

# 2. Criação da Figura: Histograma agrupado que faz a contagem automática por gênero e status
fig_sex = px.histogram(plot_df, x='Sex', color='Status', 
                 title="Taxa de Sobrevivência por Gênero", barmode='group',
                 color_discrete_map={'Não Sobreviveu': '#5E7E99', 'Sobreviveu': '#D4AF37'})

# 3. Rótulos e Interação: Configuração da formatação numérica sobre as barras e customização do tooltip (hover)
fig_sex.update_traces(
    textposition="outside",
    texttemplate="%{y:,.0f}",
    hovertemplate="<b>Gênero: %{x}</b><br>Status: %{data.name}<br>Quantidade: %{y:,.0f} passageiros<extra></extra>"
)

# 4. Escala Dinâmica: Cálculo da frequência máxima agrupada para garantir o dimensionamento correto do gráfico
max_count_sex = plot_df.groupby(["Sex", "Status"]).size().max()

# 5. Estilização do Layout: Aplicação do dark theme, formatação da legenda e ajuste de folga no eixo Y (15%)
fig_sex.update_layout(
    separators=",.",
    title=dict(x=0.5, font=dict(size=22, family="Arial", color="#E2E8F0")),
    xaxis_title="Gênero",
    yaxis_title="Quantidade de Passageiros",
    font=dict(family="Arial", size=14, color="#E2E8F0"),
    plot_bgcolor='#0B1320',
    paper_bgcolor='#0B1320',
    legend_title_font_color="#E2E8F0",
    legend_font_color="#E2E8F0",
    xaxis=dict(showgrid=False, color="#E2E8F0"),
    yaxis=dict(
        showgrid=True, 
        gridcolor="#1E293B", 
        color="#E2E8F0",
        range=[0, max_count_sex * 1.15]
    )
)

fig_sex.write_html('../site_port/graphics/titanic_fig_sex.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_sex.show()

### Interpretação: Taxa de Sobrevivência por Sexo

> **Insight:** A visualização reforça de maneira contundente que a diretriz histórica do "mulheres e crianças primeiro" foi de fato aplicada durante o naufrágio. A probabilidade condicional de sobrevivência de uma mulher a bordo era significativamente superior à probabilidade de sobrevivência de um homem.

In [ ]:
# 1. Preparação dos Dados: Criação de cópia e tradução das variáveis alvo e preditora para rótulos descritivos
plot_df_class = train_df.copy()
plot_df_class['Status'] = plot_df_class['Survived'].map({0: 'Não Sobreviveu', 1: 'Sobreviveu'})

plot_df_class['Classe_Nome'] = plot_df_class['Pclass'].map({
    1: 'Alta', 
    2: 'Média', 
    3: 'Baixa'
})

# 2. Criação da Figura: Histograma com cores customizadas e ordenação lógica forçada do eixo X (Baixa -> Média -> Alta)
fig_pclass = px.histogram(plot_df_class, x='Classe_Nome', color='Status', 
                    title="Taxa de Sobrevivência por Classe do Bilhete", barmode='group',
                    color_discrete_map={'Não Sobreviveu': '#5E7E99', 'Sobreviveu': '#D4AF37'},
                    category_orders={"Classe_Nome": ["Baixa", "Média", "Alta"]})

# 3. Rótulos e Interação: Formatação dos números sobre as barras e customização do tooltip (hover)
fig_pclass.update_traces(
    textposition="outside",
    texttemplate="%{y:,.0f}",
    hovertemplate="<b>Classe %{x}</b><br>Status: %{data.name}<br>Quantidade: %{y:,.0f} passageiros<extra></extra>"
)

# 4. Escala Dinâmica: Descoberta da barra mais alta para evitar cortes visuais no layout final
max_count_class = plot_df_class.groupby(['Classe_Nome', 'Status']).size().max()

# 5. Estilização do Layout: Aplicação do dark theme, formatação da legenda e margem extra no eixo Y (15%)
fig_pclass.update_layout(
    separators=",.",
    title=dict(x=0.5, font=dict(size=22, family="Arial", color="#E2E8F0")),
    xaxis_title="Classe",
    yaxis_title="Quantidade de Passageiros",
    font=dict(family="Arial", size=14, color="#E2E8F0"),
    plot_bgcolor='#0B1320',
    paper_bgcolor='#0B1320',
    legend_title_font_color="#E2E8F0",
    legend_font_color="#E2E8F0",
    xaxis=dict(showgrid=False, color="#E2E8F0", tickmode='linear'),
    yaxis=dict(showgrid=True, gridcolor="#1E293B", color="#E2E8F0", range=[0, max_count_class * 1.15])
)

fig_pclass.write_html('../site_port/graphics/titanic_fig_pclass.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_pclass.show()

### Interpretação: Status Socioeconômico (Classe do Bilhete)

> **Insight:** Existe uma clara desigualdade nas taxas de sobrevivência relacionada ao status socioeconômico (*Pclass*). Passageiros da 1ª classe apresentaram as maiores taxas de sobrevivência. Na 3ª classe (que concentrava a maioria dos passageiros), a grande maioria não sobreviveu, indicando que a proximidade física aos botes e o favorecimento explícito ditaram as chances de sucesso no resgate.

### 3.4. Sobrevivência por Faixa Etária

In [ ]:
# 1. Feature Engineering (Engenharia de Atributos): Discretização da variável contínua 'Age' em categorias lógicas
bins = [0, 12, 24, 59, 100]
labels = ['Crianças (0-12)', 'Jovens (13-24)', 'Adultos (25-59)', 'Idosos (60+)']
train_df['Faixa_Etaria'] = pd.cut(train_df['Age'], bins=bins, labels=labels)

# 2. Preparação dos Dados: Agrupamento e contagem para otimização do plot, seguido do mapeamento da variável Target
age_surv = train_df.groupby(['Faixa_Etaria', 'Survived'], observed=True).size().reset_index(name='Count')
age_surv['Survived'] = age_surv['Survived'].map({0: 'Não Sobreviveu', 1: 'Sobreviveu'})

# 3. Criação da Figura: Gráfico de barras agrupado mantendo a paleta de cores padronizada do projeto
fig_age = px.bar(age_surv, x='Faixa_Etaria', y='Count', color='Survived', 
                 barmode='group', title="Sobrevivência por Faixa Etária",
                 color_discrete_map={'Não Sobreviveu': '#5E7E99', 'Sobreviveu': '#D4AF37'})

# 4. Rótulos e Interação: Exibição de valores literais sobre as barras e customização do tooltip contextual (hover)
fig_age.update_traces(
    textposition="outside",
    texttemplate="%{y:,.0f}",
    hovertemplate="<b>Faixa Etária: %{x}</b><br>Status: %{data.name}<br>Quantidade: %{y:,.0f} passageiros<extra></extra>"
)

# 5. Escala Dinâmica: Identificação da contagem máxima para evitar cortes visuais no limite superior do gráfico
max_count_age = age_surv['Count'].max()

# 6. Estilização do Layout: Aplicação do dark theme, customização tipográfica e ajuste de folga no eixo Y (15%)
fig_age.update_layout(
    separators=",.",
    title=dict(x=0.5, font=dict(size=22, family="Arial", color="#E2E8F0"), pad=dict(b=15)),
    margin=dict(t=90), 
    xaxis_title="", 
    yaxis_title="Quantidade de Passageiros",
    font=dict(family="Arial", size=14, color="#E2E8F0"),
    plot_bgcolor='#0B1320',
    paper_bgcolor='#0B1320',
    legend_title_text="Status",
    legend_title_font_color="#E2E8F0",
    legend_font_color="#E2E8F0",
    xaxis=dict(showgrid=False, color="#E2E8F0"),
    yaxis=dict(
        showgrid=True, 
        gridcolor="#1E293B", 
        color="#E2E8F0",
        range=[0, max_count_age * 1.15]
    )
)

fig_age.write_html('../site_port/graphics/titanic_fig_age.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_age.show()

# 7. Prevenção de Data Leakage e Limpeza: Remoção da feature textual temporária para garantir a compatibilidade com algoritmos de Machine Learning
train_df.drop(columns=['Faixa_Etaria'], inplace=True)

### Interpretação: Taxa de Sobrevivência por Faixa Etária

> **Insight:** Observando as faixas etárias, o grupo das crianças (0-12 anos) é o único onde a barra de "Sobreviveu" supera visivelmente a de "Não Sobreviveu". Adultos e jovens tiveram as maiores baixas absolutas, confirmando que a prioridade de evacuação foi rigidamente mantida para os mais jovens.

### 3.5. Nuvem de Palavras: Sobrenomes dos Passageiros

A *Word Cloud* (Nuvem de Palavras) abaixo destaca a frequência dos sobrenomes das famílias que embarcaram. Fontes maiores indicam uma maior incidência daquele núcleo familiar a bordo (como "Andersson", "Sage" e "Goodwin"). Em tragédias históricas, grandes grupos familiares frequentemente enfrentaram maiores dificuldades de coordenação na evacuação.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 1. Engenharia de Atributos (Feature Engineering): Isolando os sobrenomes para análise de potenciais grupos familiares
# Como o padrão da coluna é "Sobrenome, Titulo. Nome", o split(',') separa no primeiro marcador e pega o índice [0]
train_df['Surname'] = train_df['Name'].apply(lambda x: x.split(',')[0].strip())

# 2. Processamento de Texto: Agregação de todos os sobrenomes em uma única string (formato exigido pela biblioteca WordCloud)
text = " ".join(train_df['Surname'].tolist())

# 3. Configuração do Objeto Visual: Parametrização da nuvem de palavras
# O colormap='YlOrBr' (Yellow-Orange-Brown) garante a harmonia visual com os gráficos anteriores
wordcloud = WordCloud(width=800, height=400, 
                      background_color='#0A1118', 
                      colormap='YlOrBr', 
                      max_words=100, 
                      contour_width=0).generate(text)

# 4. Renderização (Matplotlib): Configuração do canvas e plotagem da imagem final
plt.figure(figsize=(10, 5), facecolor='#0A1118') # facecolor garante que o fundo externo da figura acompanhe o dark theme
plt.imshow(wordcloud, interpolation='bilinear')  # Suaviza as bordas das letras para melhor legibilidade
plt.axis("off") # Oculta a grade de eixos cartesianos (que não são aplicáveis aqui)
plt.title("Nuvem de Sobrenomes dos Passageiros", color="#C19A5B", fontsize=20, fontname='Georgia', pad=20)

plt.savefig('../site_port/graphics/titanic_wordcloud.png', bbox_inches='tight', facecolor='#0A1118', dpi=300)
plt.show()

# --- Plotly Interativo Nuvem de Palavras ---
from collections import Counter
import numpy as np
import plotly.express as px
import pandas as pd

word_counts = Counter(text.split())
words_df = pd.DataFrame(word_counts.items(), columns=['Word', 'Freq']).sort_values(by='Freq', ascending=False).head(80)
np.random.seed(42)
words_df['x'] = np.random.rand(len(words_df))
words_df['y'] = np.random.rand(len(words_df))
fig_wc = px.scatter(words_df, x='x', y='y', size='Freq', color='Freq', text='Word',
                    color_continuous_scale=['#5E7E99', '#D4AF37'], title="Nuvem de Palavras Interativa (Plotly)")
fig_wc.update_traces(textposition='middle center', marker=dict(opacity=0.4), hovertemplate='<b>%{text}</b><br>Frequência: %{marker.size}<extra></extra>')
fig_wc.update_layout(xaxis=dict(showgrid=False, zeroline=False, visible=False),
                     yaxis=dict(showgrid=False, zeroline=False, visible=False),
                     plot_bgcolor='#0B1320', paper_bgcolor='#0B1320', font=dict(color='#E2E8F0', family='Georgia'),
                     coloraxis_showscale=False)
fig_wc.write_html('../site_port/graphics/titanic_wordcloud_plotly.html', include_plotlyjs='cdn', config={'displayModeBar': False})

# 5. Prevenção de Data Leakage e Limpeza: Remoção da feature textual temporária para garantir a compatibilidade com algoritmos de Machine Learning
train_df.drop(columns=['Surname'], inplace=True)

## 4. PRÉ-PROCESSAMENTO E FEATURE ENGINEERING

### 4.1. Concatenação Estratégica (Treino + Teste)
Antes de iniciarmos o tratamento de dados faltantes (imputação) e a engenharia de novas features (Feature Engineering), unimos temporariamente os datasets de treino e de teste.

> **Insight:** Essa estratégia (desde que cuidemos para não gerar *Data Leakage*) garante consistência absoluta nas transformações. Por exemplo, se aplicarmos um *One-Hot Encoding* na porta de embarque e no teste houver uma porta que não existia no treino, a concatenação prévia evita falhas dimensionais no modelo ao dividirmos novamente os conjuntos.

In [ ]:
# Concatenar treino e teste para aplicar as mesmas transformações
# Guardamos o tamanho do treino para separar novamente depois
ntrain = train_df.shape[0]
all_data = pd.concat((train_df, test_df)).reset_index(drop=True)

# 1. Extração de Títulos dos nomes
all_data['Title'] = all_data['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
all_data['Title'] = all_data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
all_data['Title'] = all_data['Title'].replace('Mlle', 'Miss')
all_data['Title'] = all_data['Title'].replace('Ms', 'Miss')
all_data['Title'] = all_data['Title'].replace('Mme', 'Mrs')

# 2. Tamanho da Família
all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1

def map_family_size(size):
    if size == 1:
        return 'Alone'
    elif size <= 4:
        return 'Small'
    else:
        return 'Large'

all_data['FamilyType'] = all_data['FamilySize'].apply(map_family_size)

# 3. Imputação de Idade (Age) com base na mediana por Title e Pclass
all_data['Age'] = all_data.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

# 4. Imputação de Fare (Tarifa) e Embarked (Embarque)
all_data['Fare'] = all_data['Fare'].fillna(all_data['Fare'].median())
all_data['Embarked'] = all_data['Embarked'].fillna(all_data['Embarked'].mode()[0])

# Dropar colunas que não usaremos nos modelos
all_data = all_data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

In [ ]:
# 5. One-Hot Encoding e Scaling
all_data = pd.get_dummies(all_data, columns=['Sex', 'Embarked', 'Title', 'FamilyType'], drop_first=True)

# Separar os dados novamente ANTES do scaler para evitar Data Leakage
X_train_full = all_data[:ntrain].drop('Survived', axis=1)
y_train_full = train_df['Survived']
X_test_final = all_data[ntrain:].drop('Survived', axis=1)

# Aplicar o Scaler estritamente nos dados numéricos
cols_to_scale = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch']
scaler = StandardScaler()

# O fit_transform acontece APENAS no treino
X_train_full[cols_to_scale] = scaler.fit_transform(X_train_full[cols_to_scale])

# O teste sofre apenas o transform (usando a média/variância aprendida no treino)
X_test_final[cols_to_scale] = scaler.transform(X_test_final[cols_to_scale])

### 4.2. Tratamento de Variáveis Categóricas e Escalonamento
Muitos algoritmos (como a Regressão Logística ou KNN) são baseados em cálculos de distância e funções lineares. Portanto, exigem que:
1. Variáveis categóricas (como 'Sex' e 'Embarked') sejam convertidas em valores numéricos utilizando técnicas como o **One-Hot Encoding**.
2. Variáveis contínuas (como 'Age' e 'Fare') estejam na mesma escala para que nenhuma domine o modelo artificialmente devido ao seu tamanho nominal. Utilizamos a **Padronização (StandardScaler)** para deixar essas distribuições com média 0 e desvio padrão 1.

## 5. TRAIN / TEST SPLIT

In [ ]:
# Divisão 80/20 para validação local
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

print("=== SHAPE FINAL DOS DADOS ===")
print(f"Treino (80%): {X_train.shape}")
print(f"Validação (20%): {X_val.shape}")
print(f"Submissão Kaggle: {X_test_final.shape}")

A **Regressão Logística** é um excelente modelo como *baseline* (modelo de referência). Apesar do nome, trata-se de um modelo de classificação que utiliza a função Sigmoide para transformar a saída de uma combinação linear das variáveis preditoras em uma probabilidade de ocorrência do evento (sobreviver ou não). É um modelo linear simples, rápido, estatisticamente rigoroso e muito interpretável.

## 6. LOGISTIC REGRESSION (Tuning)

In [ ]:
# 1. Instanciação do Modelo Base: Aumentando o limite de iterações (max_iter) para garantir que o algoritmo matemático consiga convergir, e fixando a semente (random_state)
logreg = LogisticRegression(max_iter=1000, random_state=42)

# 2. Espaço de Hiperparâmetros: Definindo a grade de testes. O parâmetro 'C' controla a força da regularização inversa (valores menores = regularização mais forte contra overfitting)
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2']}

# 3. Otimização de Busca Exaustiva: Configurando o GridSearchCV para testar absolutamente todas as combinações da grade usando Validação Cruzada de 5 partes (cv=5)
grid_lr = GridSearchCV(logreg, param_grid_lr, cv=5, scoring='accuracy')

# 4. Treinamento (Fit): Executando a busca na base de treino para encontrar o melhor ajuste linear entre as variáveis e a sobrevivência
grid_lr.fit(X_train, y_train)

# 5. Extração de Resultados: Isolando o modelo (estimador) que obteve a melhor performance média durante a validação cruzada para ser usado nas previsões finais
best_lr = grid_lr.best_estimator_
print(f"Melhores Parâmetros Logistic Regression: {grid_lr.best_params_}")

In [ ]:
# Fazendo as previsões nos dados de validação usando o melhor modelo encontrado
y_pred_lr = best_lr.predict(X_val)

# Calculando as probabilidades para a curva ROC/AUC (probabilidade da classe 1)
y_prob_lr = best_lr.predict_proba(X_val)[:, 1]

# Imprimindo o Boletim de Notas do Modelo
print("=== DESEMPENHO: LOGISTIC REGRESSION ===")
print(f"Acurácia:  {accuracy_score(y_val, y_pred_lr):.4f}")
print(f"Precisão:  {precision_score(y_val, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_val, y_pred_lr):.4f}")
print(f"F1-Score:  {f1_score(y_val, y_pred_lr):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_val, y_prob_lr):.4f}")
print("-" * 39)
print("Matriz de Confusão:")
print(confusion_matrix(y_val, y_pred_lr))

In [ ]:
# ==========================================
# PAINEL DE PERFORMANCE: LOGISTIC REGRESSION
# ==========================================
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve

fig_lr = make_subplots(rows=1, cols=3, subplot_titles=("Matriz de Confusão", "Curva ROC", "Curva Precision-Recall"))

# 1. Matriz de Confusão
cm_lr = confusion_matrix(y_val, y_pred_lr)
z_data_lr = cm_lr[::-1] 
heatmap_lr = go.Heatmap(
    z=z_data_lr, x=['Não Sobreviveu', 'Sobreviveu'], y=['Sobreviveu', 'Não Sobreviveu'],
    colorscale=['#0B1320', '#D4AF37'], showscale=False,
    text=z_data_lr, texttemplate="<b>%{text}</b>", textfont={"size":16, "color":"#E2E8F0"},
    hovertemplate="Real: %{y}<br>Predito: %{x}<br>Ocorrências: %{z}<extra></extra>"
)
fig_lr.add_trace(heatmap_lr, row=1, col=1)

# 2. Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_val, y_prob_lr)
fig_lr.add_trace(go.Scatter(x=fpr_lr, y=tpr_lr, name='ROC LR', fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', line=dict(color='#D4AF37', width=3), hovertemplate="Taxa Falsos Positivos: %{x:.2f}<br>Taxa Verdadeiros Positivos: %{y:.2f}<extra></extra>"), row=1, col=2)
fig_lr.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash', color='#5E7E99'), showlegend=False, hoverinfo='skip'), row=1, col=2)

# 3. Curva Precision-Recall
precision_lr, recall_lr, _ = precision_recall_curve(y_val, y_prob_lr)
fig_lr.add_trace(go.Scatter(x=recall_lr, y=precision_lr, name='PR LR', fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', line=dict(color='#D4AF37', width=3), hovertemplate="Recall: %{x:.2f}<br>Precision: %{y:.2f}<extra></extra>"), row=1, col=3)

# Estilização
fig_lr.update_layout(height=450, width=1200, showlegend=False, separators=",.", title=dict(text="PERFORMANCE VISUAL: LOGISTIC REGRESSION", x=0.5, font=dict(size=20, family="Arial", color="#E2E8F0")), plot_bgcolor='#0B1320', paper_bgcolor='#0B1320', font=dict(family="Arial", color="#E2E8F0"))
for annotation in fig_lr['layout']['annotations']: annotation['font'] = dict(size=14, color='#D4AF37')

# Configuração de Eixos
fig_lr.update_xaxes(title_text="Predito", row=1, col=1, title_font=dict(size=12), showgrid=False, color="#E2E8F0")
fig_lr.update_yaxes(title_text="Real", row=1, col=1, title_font=dict(size=12), showgrid=False, color="#E2E8F0")
fig_lr.update_xaxes(title_text="Taxa Falsos Positivos", range=[0, 1], row=1, col=2, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_lr.update_yaxes(title_text="Taxa Verdadeiros Positivos", range=[0, 1], row=1, col=2, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_lr.update_xaxes(title_text="Recall", range=[0, 1], row=1, col=3, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_lr.update_yaxes(title_text="Precision", range=[0, 1], row=1, col=3, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")

fig_lr.write_html('../site_port/graphics/titanic_fig_lr.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_lr.show()

Agora, utilizaremos um modelo baseado em ensemble: o **Random Forest**. Em vez de construir apenas uma árvore de decisão, ele treina múltiplas árvores (uma "floresta") usando diferentes subamostras dos dados (*Bootstrapping*) e sorteando subconjuntos de variáveis a cada nó. A previsão final é dada por votação majoritária (*Bagging*), o que o torna incrivelmente resistente a *overfitting* e capaz de capturar correlações não-lineares complexas.

## 7. RANDOM FOREST (Tuning)

In [ ]:
# 1. Instanciação do Modelo Base: Fixando a semente (random_state) para garantir reprodutibilidade dos resultados
rf = RandomForestClassifier(random_state=42)

# 2. Espaço de Hiperparâmetros: Definindo os limites de busca para controlar a complexidade das árvores e evitar overfitting
param_dist_rf = {
    'n_estimators': [100, 200, 300],    # Número de árvores que farão a "votação" final
    'max_depth': [None, 5, 10, 15],     # Profundidade máxima (evita que a árvore decore os dados de treino)
    'min_samples_split': [2, 5, 10],    # Mínimo de amostras necessárias para ramificar um nó
    'min_samples_leaf': [1, 2, 4]       # Mínimo de amostras que devem sobrar na ponta final (folha) da árvore
}

# 3. Otimização de Busca: Utilizando busca aleatória (10 iterações) para encontrar boas combinações de forma computacionalmente eficiente
rand_rf = RandomizedSearchCV(rf, param_distributions=param_dist_rf, n_iter=10, cv=5, scoring='accuracy', random_state=42)

# 4. Treinamento com Validação Cruzada: Avaliando as 10 combinações em 5 cortes diferentes (Folds) dos dados de treino
rand_rf.fit(X_train, y_train)

# 5. Extração de Resultados: Isolando o modelo vencedor (que obteve a melhor acurácia média) para uso nas previsões
best_rf = rand_rf.best_estimator_
print(f"Melhores Parâmetros Random Forest: {rand_rf.best_params_}")

In [ ]:
# Fazendo as previsões nos dados de validação
y_pred_rf = best_rf.predict(X_val)
y_prob_rf = best_rf.predict_proba(X_val)[:, 1]

# Imprimindo o Boletim de Notas do Modelo
print("=== DESEMPENHO: RANDOM FOREST ===")
print(f"Acurácia:  {accuracy_score(y_val, y_pred_rf):.4f}")
print(f"Precisão:  {precision_score(y_val, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_val, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_val, y_pred_rf):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_val, y_prob_rf):.4f}")
print("-" * 37)
print("Matriz de Confusão:")
print(confusion_matrix(y_val, y_pred_rf))

In [ ]:
# ==========================================
# PAINEL DE PERFORMANCE: LOGISTIC REGRESSION
# ==========================================
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve

fig_lr = make_subplots(rows=1, cols=3, subplot_titles=("Matriz de Confusão", "Curva ROC", "Curva Precision-Recall"))

# 1. Matriz de Confusão
cm_lr = confusion_matrix(y_val, y_pred_lr)
z_data_lr = cm_lr[::-1] 
heatmap_lr = go.Heatmap(
    z=z_data_lr, x=['Não Sobreviveu', 'Sobreviveu'], y=['Sobreviveu', 'Não Sobreviveu'],
    colorscale=['#0B1320', '#D4AF37'], showscale=False,
    text=z_data_lr, texttemplate="<b>%{text}</b>", textfont={"size":16, "color":"#E2E8F0"},
    hovertemplate="Real: %{y}<br>Predito: %{x}<br>Ocorrências: %{z}<extra></extra>"
)
fig_lr.add_trace(heatmap_lr, row=1, col=1)

# 2. Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_val, y_prob_lr)
fig_lr.add_trace(go.Scatter(x=fpr_lr, y=tpr_lr, name='ROC LR', fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', line=dict(color='#D4AF37', width=3), hovertemplate="Taxa Falsos Positivos: %{x:.2f}<br>Taxa Verdadeiros Positivos: %{y:.2f}<extra></extra>"), row=1, col=2)
fig_lr.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash', color='#5E7E99'), showlegend=False, hoverinfo='skip'), row=1, col=2)

# 3. Curva Precision-Recall
precision_lr, recall_lr, _ = precision_recall_curve(y_val, y_prob_lr)
fig_lr.add_trace(go.Scatter(x=recall_lr, y=precision_lr, name='PR LR', fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', line=dict(color='#D4AF37', width=3), hovertemplate="Recall: %{x:.2f}<br>Precision: %{y:.2f}<extra></extra>"), row=1, col=3)

# Estilização
fig_lr.update_layout(height=450, width=1200, showlegend=False, separators=",.", title=dict(text="PERFORMANCE VISUAL: LOGISTIC REGRESSION", x=0.5, font=dict(size=20, family="Arial", color="#E2E8F0")), plot_bgcolor='#0B1320', paper_bgcolor='#0B1320', font=dict(family="Arial", color="#E2E8F0"))
for annotation in fig_lr['layout']['annotations']: annotation['font'] = dict(size=14, color='#D4AF37')

# Configuração de Eixos
fig_lr.update_xaxes(title_text="Predito", row=1, col=1, title_font=dict(size=12), showgrid=False, color="#E2E8F0")
fig_lr.update_yaxes(title_text="Real", row=1, col=1, title_font=dict(size=12), showgrid=False, color="#E2E8F0")
fig_lr.update_xaxes(title_text="Taxa Falsos Positivos", range=[0, 1], row=1, col=2, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_lr.update_yaxes(title_text="Taxa Verdadeiros Positivos", range=[0, 1], row=1, col=2, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_lr.update_xaxes(title_text="Recall", range=[0, 1], row=1, col=3, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_lr.update_yaxes(title_text="Precision", range=[0, 1], row=1, col=3, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")

fig_lr.write_html('../site_port/graphics/titanic_fig_lr.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_lr.show()

A seguir, implementaremos o **XGBoost** (Extreme Gradient Boosting), que é uma implementação otimizada e avançada do método *Gradient Boosting*. Diferente da Random Forest que constrói árvores independentes (Bagging), o XGBoost constrói árvores sequencialmente, onde cada nova árvore é focada em corrigir os erros residuais da árvore anterior (*Boosting*). É notório por sua alta performance e velocidade.

## 8. XGBOOST (Tuning)

In [ ]:
# 1. Instanciação do Modelo Base: Definindo 'logloss' para evitar avisos (warnings) nas versões mais recentes do XGBoost
xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

# 2. Espaço de Hiperparâmetros: Ajuste fino do Gradient Boosting
param_dist_xgb = {
    'n_estimators': [100, 200, 300],         # Quantidade de rodadas de boosting (número de árvores sequenciais)
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # "Passo" de aprendizado: valores menores exigem mais árvores, mas generalizam melhor
    'max_depth': [3, 5, 7],                  # Profundidade da árvore (no XGBoost, árvores mais rasas costumam performar melhor do que no RF)
    'subsample': [0.8, 1.0],                 # Fração de linhas sorteadas para construir cada árvore (0.8 = 80% dos dados)
    'colsample_bytree': [0.8, 1.0]           # Fração de colunas (features) sorteadas para cada árvore, ajuda a evitar overfitting
}

# 3. Otimização de Busca: Busca aleatória para encontrar a melhor combinação com ótimo custo-benefício computacional
rand_xgb = RandomizedSearchCV(xgb_model, param_distributions=param_dist_xgb, n_iter=10, cv=5, scoring='accuracy', random_state=42)

# 4. Treinamento (Fit): Ajustando o modelo aos dados de treino validando em 5 folds (cortes)
rand_xgb.fit(X_train, y_train)

# 5. Extração de Resultados: Armazenando o estimador otimizado
best_xgb = rand_xgb.best_estimator_
print(f"Melhores Parâmetros XGBoost: {rand_xgb.best_params_}")

In [ ]:
# Fazendo as previsões nos dados de validação com o XGBoost Otimizado
y_pred_xgb = best_xgb.predict(X_val)
y_prob_xgb = best_xgb.predict_proba(X_val)[:, 1]

# Imprimindo o Boletim de Notas do Modelo
print("=== DESEMPENHO: XGBOOST ===")
print(f"Acurácia:  {accuracy_score(y_val, y_pred_xgb):.4f}")
print(f"Precisão:  {precision_score(y_val, y_pred_xgb):.4f}")
print(f"Recall:    {recall_score(y_val, y_pred_xgb):.4f}")
print(f"F1-Score:  {f1_score(y_val, y_pred_xgb):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_val, y_prob_xgb):.4f}")
print("-" * 33)
print("Matriz de Confusão:")
print(confusion_matrix(y_val, y_pred_xgb))

In [ ]:
# ==========================================
# PAINEL DE PERFORMANCE: RANDOM FOREST
# ==========================================
fig_rf = make_subplots(rows=1, cols=3, subplot_titles=("Matriz de Confusão", "Curva ROC", "Curva Precision-Recall"))

# 1. Matriz de Confusão
cm_rf = confusion_matrix(y_val, y_pred_rf)
z_data_rf = cm_rf[::-1] 
heatmap_rf = go.Heatmap(
    z=z_data_rf, x=['Não Sobreviveu', 'Sobreviveu'], y=['Sobreviveu', 'Não Sobreviveu'],
    colorscale=['#0B1320', '#D4AF37'], showscale=False,
    text=z_data_rf, texttemplate="<b>%{text}</b>", textfont={"size":16, "color":"#E2E8F0"},
    hovertemplate="Real: %{y}<br>Predito: %{x}<br>Ocorrências: %{z}<extra></extra>"
)
fig_rf.add_trace(heatmap_rf, row=1, col=1)

# 2. Curva ROC
fpr_rf, tpr_rf, _ = roc_curve(y_val, y_prob_rf)
fig_rf.add_trace(go.Scatter(x=fpr_rf, y=tpr_rf, name='ROC RF', fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', line=dict(color='#D4AF37', width=3), hovertemplate="Taxa Falsos Positivos: %{x:.2f}<br>Taxa Verdadeiros Positivos: %{y:.2f}<extra></extra>"), row=1, col=2)
fig_rf.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash', color='#5E7E99'), showlegend=False, hoverinfo='skip'), row=1, col=2)

# 3. Curva Precision-Recall
precision_rf, recall_rf, _ = precision_recall_curve(y_val, y_prob_rf)
fig_rf.add_trace(go.Scatter(x=recall_rf, y=precision_rf, name='PR RF', fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', line=dict(color='#D4AF37', width=3), hovertemplate="Recall: %{x:.2f}<br>Precision: %{y:.2f}<extra></extra>"), row=1, col=3)

# Estilização
fig_rf.update_layout(height=450, width=1200, showlegend=False, separators=",.", title=dict(text="PERFORMANCE VISUAL: RANDOM FOREST", x=0.5, font=dict(size=20, family="Arial", color="#E2E8F0")), plot_bgcolor='#0B1320', paper_bgcolor='#0B1320', font=dict(family="Arial", color="#E2E8F0"))
for annotation in fig_rf['layout']['annotations']: annotation['font'] = dict(size=14, color='#D4AF37')

# Configuração de Eixos
fig_rf.update_xaxes(title_text="Predito", row=1, col=1, title_font=dict(size=12), showgrid=False, color="#E2E8F0")
fig_rf.update_yaxes(title_text="Real", row=1, col=1, title_font=dict(size=12), showgrid=False, color="#E2E8F0")
fig_rf.update_xaxes(title_text="Taxa Falsos Positivos", range=[0, 1], row=1, col=2, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_rf.update_yaxes(title_text="Taxa Verdadeiros Positivos", range=[0, 1], row=1, col=2, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_rf.update_xaxes(title_text="Recall", range=[0, 1], row=1, col=3, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
fig_rf.update_yaxes(title_text="Precision", range=[0, 1], row=1, col=3, showgrid=True, gridcolor="#1E293B", color="#E2E8F0")

fig_rf.write_html('../site_port/graphics/titanic_fig_rf.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_rf.show()

## 9. COMPARAÇÃO DOS MODELOS E EXPORTAÇÃO

In [ ]:
# 1. Agrupamento de Modelos: Criando um dicionário com os estimadores já otimizados para facilitar a automação
models = {
    'Logistic Regression': best_lr,
    'Random Forest': best_rf,
    'XGBoost': best_xgb
}

# 2. Inicialização: Lista vazia que atuará como um repositório temporário para o "boletim de notas"
results = []

# 3. Loop de Avaliação: Iterando sistematicamente sobre cada modelo para avaliar o desempenho
for name, model in models.items():
    # Geração de previsões absolutas (0 ou 1) e probabilísticas (necessárias para o cálculo do ROC-AUC)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    # Cálculo das métricas de classificação
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    # Armazenando o resultado da iteração atual
    results.append([name, acc, prec, rec, f1, auc])

# 4. Consolidação Analítica: Estruturando os resultados em um DataFrame ordenado da maior para a menor Acurácia
results_df = pd.DataFrame(results, columns=['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])
results_df = results_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

print("=== COMPARAÇÃO DOS MODELOS ===")
display(results_df)

### 9.1. Interpretação dos Resultados dos Modelos

> **Insight:** Ao avaliarmos diferentes algoritmos de Machine Learning, não devemos olhar apenas para a **Acurácia**.
> * **ROC-AUC:** É uma métrica excelente para avaliar o poder do modelo em distinguir as classes independentemente do limiar (threshold).
> * **F1-Score:** Como nossas classes têm um leve desbalanceamento (muito mais não sobreviventes do que sobreviventes), o F1-Score (média harmônica entre Precision e Recall) nos dá uma visão mais realista do desempenho nas predições das duas classes.
> 
> O algoritmo em formato de *Ensemble* (Random Forest ou XGBoost) geralmente apresenta métricas superiores devido à capacidade natural dessas árvores de capturar relações não-lineares nos dados (como a combinação de Classe + Idade + Sexo).

### 9.2. Painel Visual Comparativo: Matrizes e Curvas de Performance

Existem três visualizações que representam o "padrão ouro" para avaliação de modelos de classificação:

1. **Matriz de Confusão:** Uma tabela que cruza os valores **Reais** com as **Predições**. O ideal é uma diagonal principal forte (Verdadeiros Positivos e Verdadeiros Negativos), enquanto os erros (Falsos Positivos e Falsos Negativos) devem tender a zero.
2. **Curva ROC (Receiver Operating Characteristic):** Plota a Taxa de Verdadeiros Positivos contra a Taxa de Falsos Positivos em diferentes limiares de decisão. Quanto mais próxima a linha estiver do canto superior esquerdo (AUC = 1.0), melhor o modelo. Uma linha diagonal representa a mesma sorte de jogar uma moeda (AUC = 0.50).
3. **Curva Precision-Recall (PR):** Especialmente importante para datasets com **classes desbalanceadas** (como o nosso, onde mais pessoas faleceram do que sobreviveram). Ela foca puramente no desempenho sobre a classe positiva (os sobreviventes). Quanto mais próxima do canto superior direito (AUC = 1.0), melhor.

> **Insight:** Analisando o painel abaixo, note que o Random Forest e o XGBoost normalmente superam a Regressão Logística tanto em termos de curvas (área sob a curva) quanto ao minimizar Falsos Positivos (visíveis na Matriz de Confusão).

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, roc_curve, roc_auc_score
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Arquitetura do Dashboard: Criando uma grade 3x3 (um modelo por linha, uma métrica por coluna)
fig = make_subplots(rows=3, cols=3, 
                    subplot_titles=(
                        "LogReg - Matriz", "LogReg - ROC", "LogReg - PR",
                        "RandomForest - Matriz", "RandomForest - ROC", "RandomForest - PR",
                        "XGBoost - Matriz", "XGBoost - ROC", "XGBoost - PR"
                    ))

# 2. Loop Analítico: Iterando dinamicamente sobre o dicionário de modelos otimizados para evitar duplicação de código
for i, (name, model) in enumerate(models.items(), start=1):
    # Extração das probabilidades e classes preditas
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    # --- COLUNA 1: Matriz de Confusão ---
    cm = confusion_matrix(y_val, y_pred)
    z_data = cm[::-1] # Inverte a matriz verticalmente para o Plotly renderizar na ordem correta (Real em Y, Predito em X)
    y_labels = ['Sobreviveu', 'Não Sobreviveu']
    x_labels = ['Não Sobreviveu', 'Sobreviveu']
    
    heatmap = go.Heatmap(
        z=z_data, x=x_labels, y=y_labels,
        colorscale=['#0B1320', '#D4AF37'], # Mantém a identidade visual: Fundo escuro para baixo volume, Dourado para alto volume
        showscale=False, # Oculta a barra de cores lateral para um visual mais limpo
        text=z_data, texttemplate="<b>%{text}</b>", textfont={"size":16, "color":"#E2E8F0"},
        hovertemplate="Real: %{y}<br>Predito: %{x}<br>Ocorrências: %{z}<extra></extra>"
    )
    fig.add_trace(heatmap, row=i, col=1)
    
    # --- COLUNA 2: Curva ROC (Receiver Operating Characteristic) ---
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'ROC {name}', 
                             fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)', # Sombra dourada translúcida sob a curva
                             line=dict(color='#D4AF37', width=3),
                             hovertemplate="Taxa Falsos Positivos: %{x:.2f}<br>Taxa Verdadeiros Positivos: %{y:.2f}<extra></extra>"), 
                  row=i, col=2)
    # Linha de base (modelo aleatório): Se a curva do modelo ficar abaixo dessa linha pontilhada, ele é pior que o acaso
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash', color='#5E7E99'), showlegend=False, hoverinfo='skip'), row=i, col=2)
    
    # --- COLUNA 3: Curva Precision-Recall (PR) ---
    precision, recall, _ = precision_recall_curve(y_val, y_prob)
    fig.add_trace(go.Scatter(x=recall, y=precision, name=f'PR {name}', 
                             fill='tozeroy', fillcolor='rgba(212, 175, 55, 0.2)',
                             line=dict(color='#D4AF37', width=3),
                             hovertemplate="Recall: %{x:.2f}<br>Precision: %{y:.2f}<extra></extra>"), 
                  row=i, col=3)

# 3. Estilização Global: Aplicação do Dark Theme e definição de dimensões adequadas para um painel com 9 gráficos
fig.update_layout(
    height=1000, width=1200, showlegend=False, separators=",.",
    title=dict(text="PAINEL COMPARATIVO DE PERFORMANCE DOS MODELOS", x=0.5, font=dict(size=24, family="Arial", color="#E2E8F0")),
    plot_bgcolor='#0B1320',
    paper_bgcolor='#0B1320',
    font=dict(family="Arial", color="#E2E8F0")
)

# 4. Refinamento Tipográfico: Destacando os títulos individuais dos subplots com a cor de destaque (Dourado)
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=14, color='#D4AF37')

# 5. Padronização dos Eixos: Otimizando a leitura aplicando a grade de fundo e limitando os eixos de probabilidade de 0 a 1
for i in range(1, 4):
    # Eixos das Matrizes de Confusão (Col 1)
    fig.update_xaxes(title_text="Predito", row=i, col=1, title_font=dict(family="Arial", size=12, color="#E2E8F0"), showgrid=False, color="#E2E8F0")
    fig.update_yaxes(title_text="Real", row=i, col=1, title_font=dict(family="Arial", size=12, color="#E2E8F0"), showgrid=False, color="#E2E8F0")
    
    # Eixos das Curvas ROC (Col 2)
    fig.update_xaxes(title_text="Taxa Falsos Positivos", range=[0, 1], row=i, col=2, title_font=dict(family="Arial", size=12, color="#E2E8F0"), showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
    fig.update_yaxes(title_text="Taxa Verdadeiros Positivos", range=[0, 1], row=i, col=2, title_font=dict(family="Arial", size=12, color="#E2E8F0"), showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
    
    # Eixos das Curvas PR (Col 3)
    fig.update_xaxes(title_text="Recall", range=[0, 1], row=i, col=3, title_font=dict(family="Arial", size=12, color="#E2E8F0"), showgrid=True, gridcolor="#1E293B", color="#E2E8F0")
    fig.update_yaxes(title_text="Precision", range=[0, 1], row=i, col=3, title_font=dict(family="Arial", size=12, color="#E2E8F0"), showgrid=True, gridcolor="#1E293B", color="#E2E8F0")

fig.write_html('../site_port/graphics/titanic_fig_comparative.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig.show()

In [ ]:
# 1. Seleção Dinâmica: Identifica automaticamente o modelo vencedor da etapa anterior para extração de insights
best_model_name = results_df.loc[0, 'Modelo']
best_model = models[best_model_name]

# 2. Verificação de Compatibilidade: Garante que o algoritmo possui o atributo de importância (modelos lineares puros não possuem feature_importances_)
if hasattr(best_model, 'feature_importances_'):
    # Extração dos pesos e mapeamento com os nomes das colunas de treino
    importances = best_model.feature_importances_
    features = X_train.columns
    
    # Estruturação em DataFrame e ordenação crescente para o gráfico de barras horizontais
    feat_imp_df = pd.DataFrame({'Feature': features, 'Importance': importances})
    feat_imp_df = feat_imp_df.sort_values('Importance', ascending=True)
    
    # 3. Criação da Figura: Gráfico de barras horizontais com escala de cor contínua vinculada ao valor da importância
    fig_imp = px.bar(feat_imp_df, x='Importance', y='Feature', orientation='h',
                     title=f"Importância das Variáveis - {best_model_name}",
                     color='Importance',
                     color_continuous_scale=['#5E7E99', '#D4AF37'],
                     text='Importance') # Informa ao Plotly qual coluna usar para o texto interno
    
    # 4. Rótulos e Interação: Configuração do texto dentro das barras (2 casas decimais) e customização do tooltip
    fig_imp.update_traces(
        textposition="inside",
        texttemplate="%{x:,.2f}", # '%{x}' pois em barras horizontais o valor numérico está no eixo X
        hovertemplate="<b>Feature: %{y}</b><br>Peso de Importância: %{x:,.2f}<extra></extra>"
    )
    
    # 5. Estilização do Layout: Aplicação do dark theme, ocultação da barra lateral de cores (redundante) e formatação de eixos
    fig_imp.update_layout(
        separators=",.", # Ativa a formatação local para casas decimais
        title=dict(x=0.5, font=dict(size=22, family="Arial", color="#E2E8F0")),
        font=dict(family="Arial", size=14, color="#E2E8F0"),
        plot_bgcolor='#0B1320',
        paper_bgcolor='#0B1320',
        xaxis=dict(showgrid=True, gridcolor="#1E293B", color="#E2E8F0"),
        yaxis=dict(showgrid=False, color="#E2E8F0"),
        coloraxis_showscale=False # Remove a legenda lateral da escala de cores para um visual mais limpo
    )
    
    fig_imp.write_html('../site_port/graphics/titanic_fig_imp.html', include_plotlyjs='cdn', config={'displayModeBar': False})
fig_imp.show()

### 9.3. Interpretação das Variáveis Mais Importantes

> **Insight:** A análise de *Feature Importances* revela quais características tiveram mais peso na decisão do modelo. 
> O título dos passageiros (extraído do nome) e a variável "Sex" costumam dominar a predição. O alto peso atribuído a categorias como "Mr" ou "Sex_male" (associado a menores taxas de sobrevivência) reforça estatisticamente a regra social de resgate priorizando mulheres e crianças. Variáveis financeiras e socioeconômicas (como "Fare" e "Pclass") aparecem como secundárias, mas também muito influentes.

In [ ]:
# ==========================================
# CÉLULA FINAL: GERANDO A SUBMISSÃO KAGGLE
# ==========================================

# 1. Fazendo as previsões nos dados oficiais de teste usando o XGBoost Otimizado
previsoes_finais = best_xgb.predict(X_test_final)

# 2. Criando o DataFrame no formato exato exigido pela competição
submissao = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': previsoes_finais
})

# 3. Exportando para CSV (index=False é crucial para o Kaggle não recusar o arquivo)
nome_arquivo = 'submission.csv'
submissao.to_csv(nome_arquivo, index=False)

print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")
print(f"Formato: {submissao.shape[0]} linhas e {submissao.shape[1]} colunas.")

# Exibindo as 5 primeiras linhas para conferência final
display(submissao.head())